# How to work with measurement errors in regression

This notebook is about the derivation of linear regression, weighted linear regression and should serve as a cookbook for proper fitting with measurement errors with experiments and their propagation into parameter errors. 

## Content
* [Linear regression without any measurement errors](#lin_no)
* [Linear regression with measurement error for $y$](#lin_y)
* [Linear regression with measurement error for $y$ and $x_i$](#lin_y_x)
* [Deming regression with measurement errors in $y$ and $x_i$](#deming_y_x)
* [Nonlinear regression with measurement errors in $y$](#nonlin_y)
* [Nonlinear regression with measurement errors in $y$ and $x_i$](#nonlin_y_x)
* [Monte Carlo estimation of parameters errors with measurement errors for $y$ and $x_i$](#mc_y_x)
* [How to cite](#cite)

## Motivation

I studied physics and back in 2018 - 2021 and I had 5 semesters of lab measurements in total as part of my bachelor and master program. I was puzzled many times by how the errors should be correctly propagated when fitting the data. Later I found out that measurement errors can be propagated with Monte Carlo, but looking back at that time I still got some parts incorrectly. I hope that this notebook will help anyone who is trying to propagate errors from measurement into the estimated parameters. Besides that, you should also change the objective function so that it has weights calculated from those measurement errors.

## Naming and definitions

For people that never did any measurement in labs the following may be confusing. That's why I want to clarify the following
* **measurement** - Values $(x_i, y_i)$ that are sampled or measured at one time. Those values might have **measurement error** which is $(\sigma_{x_i}, \sigma_{y_i})$. We call $(x_i, y_i)$ and if given $(\sigma_{x_i}, \sigma_{y_i})$ a measurement in an experiment.
* **measurement error** - This is not mathematically well defined term, so what is it? When we measure in lab the independent variable $x$ and dependent variable $y$ the instruments may have some specified relative and absolute error. Therefore we might get an error for $x$ and $y$. Most of the time we assume that such errors are $\sigma$ of normal distributions. That means that with measurement $(x_i, y_i)$ and measurement error $(\sigma_{x_i}, \sigma_{y_i})$ we now know that the true values $(x_i, y_i)$ can be sampled from normal distributions $(\mathcal{N}(x_i,\,\sigma_{x_i}^{2}), \mathcal{N}(y_i,\,\sigma_{y_i}^{2}))$.
* **model error** - This error is caused primary by the fact that the model is too simple to model the real data well enough. In this notebook we talk about $\varepsilon \sim \mathcal{N}(0,\,\sigma^{2})$ as model error.


<a id='lin_no'></a>
## Linear regression without any measurement errors

Linear regression is a statistical model that estimates the relationship between dependent variable $y$ and independent variables $x_i$, since this relationship may be not perfect we assume small error $\varepsilon \sim \mathcal{N}(0,\,\sigma^{2})$. This error is not known during the measurement
$$y = \beta_{0} + \beta_{1} x_{1} + \cdots + \beta_{p} x_{p} + \varepsilon.$$
The error shown here, $\varepsilon$ is not a measurement error. Measurement error is something we know when we make the measurement. This error $\varepsilon$ is error of the model as the model might be an approximation and the underlying function $y=f(x)$ might be much more complex. Then we have dataset $D=\{x_{i1}, \ldots, x_{ip}, y_i\}_{i=1}^n$, where $i$ is one observation or measurement. We want to estimate $\boldsymbol{\beta}$ to get the best estimate $\hat{y}_i$
$$\hat{y}_i = \beta_{0} + \beta_{1} x_{i1} + \cdots + \beta_{p} x_{ip} = \mathbf{x}^\mathsf{T}_i\boldsymbol\beta.$$
We will often use notation 
$$Y = X^\textsf{T}\boldsymbol{\beta} + \boldsymbol{\varepsilon}.$$
How do we define what is the best estimate of $\hat{y}_i$ and hence what parameters $\boldsymbol{\beta}$ are the best? We can derive it simply. First, we will work out what is the probability of $\boldsymbol{\beta}$ given single data point $(\mathbf{x}, y_i)$ and given Bayes' theorem
$$P(\boldsymbol{\beta}| \mathbf{x}_i, y_i) = \frac{P(\mathbf{x}_i, y_i | \boldsymbol{\beta}) P(\boldsymbol{\beta}) }{P(\mathbf{x}_i, y_i)}.$$
In this notebook we assume that $P(\boldsymbol{\beta})$ is uniform and hence it doesn't have to be specified. The distribution $P(\mathbf{x}_i, y_i)$ is same for all measured data so again, we don't have to work with that neither. We end up with 
$$P(\boldsymbol{\beta}| \mathbf{x}_i, y_i) \sim P^\prime(\boldsymbol{\beta}| \mathbf{x}_i, y_i) = P(\mathbf{x}_i, y_i | \boldsymbol{\beta}),$$
where the probability $P^\prime(\mathbf{x}_i, y_i | \boldsymbol{\beta})$ is given by normal distribution
$$P^\prime(\boldsymbol{\beta}| \mathbf{x}_i, y_i) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp{\left(-\frac{(y_i - f(\mathbf{x}_i, \boldsymbol{\beta}))^2}{2\sigma^2}\right)}.$$
Similarly, the probability of having $\boldsymbol{\beta}$ given the whole dataset is
$$P(\boldsymbol{\beta}| D) \sim P^\prime(\boldsymbol{\beta}| D) = \prod_{i=1}^{N} P^\prime(\boldsymbol{\beta}| \mathbf{x}_i, y_i)$$
$$P^\prime(\boldsymbol{\beta}| D) = \prod_{i=1}^{N} \frac{1}{\sqrt{2\pi\sigma^2}}\exp{\left(-\frac{(y_i - f(\mathbf{x}_i, \boldsymbol{\beta}))^2}{2\sigma^2}\right)}.$$
In order to express $P^\prime(\boldsymbol{\beta}| D)$ as product of probabilities $P^\prime(\boldsymbol{\beta}| \mathbf{x}_i, y_i)$ we have to assume that measurements during experiment were independent. Example of not independent measurements are when you measure a temperature of gel, but you perform your measurements often and your thermometer is always at room temperature. The surrounding of thermometer will not have enough time to thermalize at the average temperature of gel. Hence every measurement of temperature will be correlated with the previous measured temperature.
Using maximum likelihood estimation (MLE) we want to change $\boldsymbol{\beta}$ so that the probability $P(\boldsymbol{\beta}| D)$ is maximized. Maximizing $P(\boldsymbol{\beta}| D)$ is equivalent to maximizing $P^\prime(\boldsymbol{\beta}| D)$. MLE is a method of estimating the parameters of an assumed probability distribution, given some observed data. We define loss function
$$\mathcal{L}(\boldsymbol{\beta}) = \log{P^\prime}(\boldsymbol{\beta}| D) = \sum_{i=1}^{N} \log P^\prime(\boldsymbol{\beta}| \mathbf{x}_i, y_i) = -\frac{N}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2} \sum_{i=1}^{N} (y_i - f(\mathbf{x}_i, \boldsymbol{\beta}))^2$$
Maximizing loss function we will find optimal parameters $\boldsymbol{\beta}$
$$\boldsymbol{\hat{\beta}} = \underset{\boldsymbol{\beta}}{\operatorname{argmax}} \mathcal{L}(\boldsymbol{\beta}) = \underset{\boldsymbol{\beta}}{\operatorname{argmax}} \left( -\frac{N}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2} \sum_{i=1}^{N} (y_i - f(\mathbf{x}_i, \boldsymbol{\beta}))^2 \right).$$
Maximizing loss function leads to least squares method
$$\boldsymbol{\hat{\beta}} = \underset{\boldsymbol{\beta}}{\operatorname{argmin}} \sum_{i=1}^{N} (y_i - f(\mathbf{x}_i, \boldsymbol{\beta}))^2 $$
We successfully derived least squares method with assumption of no measurement errors. This equation can be solved exactly, remember $\boldsymbol{\beta}$ is a vector, $Y$ is a vector and $X$ is a matrix
$$S(\boldsymbol{\beta}) = |X\boldsymbol{\beta} - Y|^2$$
$$S(\boldsymbol{\beta}) = \left(X\boldsymbol{\beta} - Y\right)^\textsf{T} \left(X\boldsymbol{\beta} - Y\right)$$
$$S(\boldsymbol{\beta}) = Y^\textsf{T}Y - Y^\textsf{T}X\boldsymbol{\beta} - \boldsymbol{\beta}^\textsf{T}X^\textsf{T} Y + \boldsymbol{\beta}^\textsf{T}X^\textsf{T} X \boldsymbol{\beta}$$
after we expanded the least squares, we can differentiate them by our parameters
$$\frac{\partial S(\boldsymbol{\beta})}{\partial\boldsymbol{\beta}} = -2X^\textsf{T}Y + 2X^\textsf{T}X\boldsymbol{\beta}.$$
For an optimal solution we want these partial derivatives to be zero
$$-2X^\textsf{T}Y + 2X^\textsf{T}X\boldsymbol{\hat{\beta}} = 0$$
$$X^\textsf{T}X\boldsymbol{\hat{\beta}} = X^\textsf{T}Y$$
$$\boldsymbol{\hat{\beta}} = \left(X^\textsf{T}X\right)^{-1}X^\textsf{T}Y.$$

We can use this solution which is implemented in for example `np.polyfit` or we can implement the optimization ourselves with a model, objective function and jacobian function for `'Newton-CG` method and use the optimization function from `scipy.optimize.minimize`.

Sources:
* https://en.wikipedia.org/wiki/Linear_regression
* https://en.wikipedia.org/wiki/Maximum_likelihood_estimation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

def model(x, a, b):
    return a * x + b

def objective_function(params, x, y):
    a, b = params
    return np.sum((y - model(x, a, b)) ** 2)

def jac_function(params, x, y):
    a, b = params
    residual = y - model(x, a, b)
    da = -2 * np.sum(x * residual)
    db = -2 * np.sum(residual)
    return np.array([da, db])

a_true = 2
b_true = 1
sigma = 0.5
x_data = np.arange(1, 11)
np.random.seed(0)
err = np.random.normal(0, sigma, len(x_data))
# we will create observed/measured data
x_obs = x_data
y_obs = model(x_data, a_true, b_true) + err

initial_guess = [0.0, 0.0]
result = minimize(
    fun=lambda params: objective_function(params, x_obs, y_obs), 
    x0=initial_guess, 
    method='Newton-CG', 
    jac=lambda params: jac_function(params, x_obs, y_obs),
    options={'disp': True, 'xtol': 1e-8}
)
a_fit, b_fit = result.x
print(f"Parameter a: {a_true:.4f}, estimated: {a_fit:.4f}")
print(f"Parameter b: {b_true:.4f}, estimated: {b_fit:.4f}")

x_fit = np.linspace(min(x_data), max(x_data), 100)
y_fit = model(x_fit, a_fit, b_fit)
y_true = model(x_fit, a_true, b_true)

plt.scatter(x_obs, y_obs, marker='o', label='Data')
plt.plot(x_fit, y_fit, label='Fitted line', color='red')
plt.plot(x_fit, y_true, label='True function', color='blue')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()


How do we estimate the error of fitted parameters? For that, we have [covariance matrix](https://en.wikipedia.org/wiki/Covariance_matrix), in our case it is just a variance matrix, despite that the term covariance matrix is used often for parameter error estimation in regression. We will recollect our optimal solution
$$\boldsymbol{\hat{\beta}} = \left(X^\textsf{T}X\right)^{-1}X^\textsf{T}Y.$$
The variance matrix for those optimal parameters is
$$\textrm{Var}(\boldsymbol{\hat{\beta}})
= (X^\textsf{T} X)^{-1} X^\textsf{T} \;\sigma^2 I \; X  (X^\textsf{T} X)^{-1}
= \sigma^2 (X^\textsf{T} X)^{-1} (X^\textsf{T} X)  (X^\textsf{T} X)^{-1}
= \sigma^2  (X^\textsf{T} X)^{-1}.$$
Remember the whole matrix $X$ has the following structure
$$
X = \begin{bmatrix} \mathbf{x}^\mathsf{T}_1 \\ \mathbf{x}^\mathsf{T}_2 \\ \vdots \\ \mathbf{x}^\mathsf{T}_n \end{bmatrix}
= \begin{bmatrix} 1 &  x_{11} & \cdots & x_{1p} \\
1 & x_{21} & \cdots & x_{2p} \\
\vdots & \vdots & \ddots & \vdots \\
1 & x_{n1} & \cdots & x_{np}
\end{bmatrix},
$$
but in fact the order of columns and rows doesn't matter as long as you are consistent in defining $X$ and $Y$.

Sources:
* https://en.wikipedia.org/wiki/Covariance_matrix
* https://stats.stackexchange.com/questions/44838/how-are-the-standard-errors-of-coefficients-calculated-in-a-regression

In [ ]:
from scipy.stats import t

residuals = y_obs - model(x_obs, a_fit, b_fit)
# we have two degrees of freedom less as we have two paramaters
mse = np.sum(residuals**2) / (len(x_obs) - 2)
X_design = np.vstack((np.ones(len(x_obs)), x_obs)).T
cov_matrix = mse * np.linalg.inv(X_design.T @ X_design)
std_errors = np.sqrt(np.diag(cov_matrix))
# we have two degrees of freedom less as we have two paramaters
df = len(x_obs) - len(initial_guess)

# t-value for 95% CI
t_value = t.ppf(1 - 0.025, df)
# 95% confidence intervals
ci_95 = t_value * std_errors
print(f"Standard error of a: ({a_fit - std_errors[1]:.4f}, {a_fit + std_errors[1]:.4f})")
print(f"95% CI for a: ({a_fit - ci_95[1]:.4f}, {a_fit + ci_95[1]:.4f})")

print(f"Standard error of b: ({b_fit - std_errors[0]:.4f}, {b_fit + std_errors[0]:.4f})")
print(f"95% CI for b: ({b_fit - ci_95[0]:.4f}, {b_fit + ci_95[0]:.4f})")

<a id='lin_y'></a>
## Linear regression with measurement error for $y$

We no longer assume error of the model $\varepsilon \sim \mathcal{N}(0,\,\sigma^{2})$ from the case where we have no measurement error. This approximation is valid if $\sigma << \sigma_{y_i}$ which is reasonable to assume. Our dataset now is still the same $D=\{x_{i1}, \ldots, x_{ip}, y_i\}_{i=1}^n$ but we have errors $D_\sigma = \{\sigma_{y_i}\}_{i=1}^n$. The whole trick is to convert this problem into well well-known one which is weighted regression. We will show briefly how to derive the loss function and optimal solution. We will start with the conditional probability for one measurement as before
$$P(\boldsymbol{\beta}| \mathbf{x}_i, y_i, \sigma_{y_i}) = \frac{1}{\sqrt{2\pi\sigma_{y_i}^2}}\exp{\left(-\frac{(y_i - f(\mathbf{x}_i, \boldsymbol{\beta}))^2}{2\sigma_{y_i}^2}\right)}.$$
Similarly, the probability of having $\boldsymbol{\beta}$ given the whole dataset is
$$P(\boldsymbol{\beta}| D) = \prod_{i=1}^{N} \frac{1}{\sqrt{2\pi\sigma_{y_i}^2}}\exp{\left(-\frac{(y_i - f(\mathbf{x}_i, \boldsymbol{\beta}))^2}{2\sigma_{y_i}^2}\right)}.$$
The alternated loss function with weights is then
$$\mathcal{L}(\boldsymbol{\beta}) = \sum_{i=1}^{N} -\frac{1}{2}\log(2\pi\sigma_{y_i}^2) - \frac{1}{2\sigma_{y_i}^2}  (y_i - f(\mathbf{x}_i, \boldsymbol{\beta}))^2$$
Maximizing loss function we will find optimal parameters $\boldsymbol{\beta}$ and this leads to weighted least squares method
$$\boldsymbol{\hat{\beta}} = \underset{\boldsymbol{\beta}}{\operatorname{argmin}} \sum_{i=1}^{N} \frac{1}{\sigma_{y_i}^2} (y_i - f(\mathbf{x}_i, \boldsymbol{\beta}))^2 $$
We successfully derived the least squares method with measurement error in variable $y$. This equation can be again solved exactly, remember $\boldsymbol{\beta}$ is a vector, $Y$ is a vector and $X$ is a matrix. We will define the weight matrix $W$ defined as 
$$W_{ii} = 1 / \sigma_{y_i}^2$$
Now we will expand weighted least squares
$$S(\boldsymbol{\beta}) = \left(X\boldsymbol{\beta} - Y\right)^\textsf{T} W \left(X\boldsymbol{\beta} - Y\right)$$
$$S(\boldsymbol{\beta}) = Y^\textsf{T}WY - Y^\textsf{T}WX\boldsymbol{\beta} - \boldsymbol{\beta}^\textsf{T}X^\textsf{T}W Y + \boldsymbol{\beta}^\textsf{T}X^\textsf{T} W X\boldsymbol{\beta}$$
after we expanded least squares, we can differentiate them by our parameters
$$\frac{\partial S(\boldsymbol{\beta})}{\partial\boldsymbol{\beta}} = -2X^\textsf{T}WY + 2X^\textsf{T}WX\boldsymbol{\beta}.$$
For an optimal solution we want these partial derivatives to be zero
$$-2X^\textsf{T}WY + 2X^\textsf{T}WX\boldsymbol{\hat{\beta}} = 0$$
$$X^\textsf{T}WX\boldsymbol{\hat{\beta}} = X^\textsf{T}WY$$
$$\boldsymbol{\hat{\beta}} = \left(X^\textsf{T}WX\right)^{-1}X^\textsf{T}WY.$$

We can use this solution which is implemented in for example `np.polyfit` or we can implement the optimization ourselves with a model, objective function and jacobian function for `'Newton-CG` method and use the optimization function from `scipy.optimize.minimize`.

Sources:
https://en.wikipedia.org/wiki/Weighted_least_squares

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

def model(x, a, b):
    return a * x + b

def objective_function(params, x, y, sigma_y):
    a, b = params
    residuals = y - model(x, a, b)
    weighted_residuals = residuals / sigma_y
    return np.sum(weighted_residuals ** 2)

def jac_function(params, x, y, sigma_y):
    a, b = params
    residuals = y - model(x, a, b)
    weighted_residuals = residuals / sigma_y
    da = -2 * np.sum(x * weighted_residuals / sigma_y)
    db = -2 * np.sum(weighted_residuals / sigma_y)
    return np.array([da, db])

a_true = 2
b_true = 1
x_data = np.arange(1, 11)

sigma_y = 0.4 * x_data 
np.random.seed(0)
err_y = np.random.normal(0, sigma_y, len(x_data))
# we will create observed/measured data
x_obs = x_data
y_obs = model(x_data, a_true, b_true) + err_y

initial_guess = [0.0, 0.0]
result = minimize(
    fun=lambda params: objective_function(params, x_obs, y_obs, sigma_y),
    x0=initial_guess, 
    method='Newton-CG', 
    jac=lambda params: jac_function(params, x_obs, y_obs, sigma_y), 
    options={'disp': True, 'xtol': 1e-8}
)
a_fit, b_fit = result.x
print(f"Parameter a: {a_true:.4f}, estimated: {a_fit:.4f}")
print(f"Parameter b: {b_true:.4f}, estimated: {b_fit:.4f}")

x_fit = np.linspace(min(x_data), max(x_data), 100)
y_fit = model(x_fit, a_fit, b_fit)
y_true = model(x_fit, a_true, b_true)

plt.errorbar(x_obs, y_obs, yerr=sigma_y, fmt='o', label='Noisy data')
plt.plot(x_fit, y_fit, label='Fitted line', color='red')
plt.plot(x_fit, y_true, label='True function', color='blue')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()


How do we estimate the error of fitted parameters when we have measurement error in $y$? We will again use a variance matrix of $\boldsymbol{\beta}$ leading to slightly different results. We define matrix $M$ as
$$M_{ii} = \sigma_{y_i}^2.$$
Remember the definition of matrix $W$
$$W_{ii} = 1 / \sigma_{y_i}^2.$$
The variance matrix for those optimal parameters is
$$\textrm{Var}(\boldsymbol{\hat{\beta}})
= (X^\textsf{T} W X)^{-1} X^\textsf{T} W M W X (X^\textsf{T} W X)^{-1}
= (X^\textsf{T} W X)^{-1} (X^\textsf{T} W X)  (X^\textsf{T} W X)^{-1}
= (X^\textsf{T} W X)^{-1}.$$

In [ ]:
from scipy.stats import t

residuals = y_obs - model(x_obs, a_fit, b_fit)
W = np.diag(1 / sigma_y**2)
X_design = np.vstack((np.ones(len(x_obs)), x_obs)).T
cov_matrix = np.linalg.inv(X_design.T @ W @ X_design)
std_errors = np.sqrt(np.diag(cov_matrix))
df = len(x_obs) - len(initial_guess)

# t-value for 95% CI
t_value = t.ppf(1 - 0.025, df)
# 95% confidence intervals
ci_95 = t_value * std_errors
print(f"Standard error of a: ({a_fit - std_errors[1]:.4f}, {a_fit + std_errors[1]:.4f})")
print(f"95% CI for a: ({a_fit - ci_95[1]:.4f}, {a_fit + ci_95[1]:.4f})")

print(f"Standard error of b: ({b_fit - std_errors[0]:.4f}, {b_fit + std_errors[0]:.4f})")
print(f"95% CI for b: ({b_fit - ci_95[0]:.4f}, {b_fit + ci_95[0]:.4f})")

You may argue that adding weights to the loss function will not change anything. This section proposes that this is an important step and should not be ignored especially when the variance of measurement errors is great. Here we will optimize the parameters without weights and we can observe that the parameters are estimated worse than with weights.

In [ ]:
def objective_function(params, x, y):
    a, b = params
    residuals = y - model(x, a, b)
    return np.sum(residuals ** 2)

def jac_function(params, x, y):
    a, b = params
    residuals = y - model(x, a, b)
    da = -2 * np.sum(x * residuals)
    db = -2 * np.sum(residuals)
    return np.array([da, db])

result = minimize(
    fun=lambda params: objective_function(params, x_obs, y_obs), 
    x0=initial_guess,  
    method='Newton-CG', 
    jac=lambda params: jac_function(params, x_obs, y_obs),
    options={'disp': True, 'xtol': 1e-8}
)
a_fit, b_fit = result.x
print(f"Parameter a: {a_true:.4f}, estimated: {a_fit:.4f}")
print(f"Parameter b: {b_true:.4f}, estimated: {b_fit:.4f}")

x_fit = np.linspace(min(x_data), max(x_data), 100)
y_fit = model(x_fit, a_fit, b_fit)
y_true = model(x_fit, a_true, b_true)

plt.errorbar(x_obs, y_obs, yerr=sigma_y, fmt='o', label='Noisy data')
plt.plot(x_fit, y_fit, label='Fitted line', color='red')
plt.plot(x_fit, y_true, label='True function', color='blue')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

<a id='lin_y_x'></a>
## Linear regression with measurement error for $y$ and $x_i$

We no longer assume error of the model  $\varepsilon \sim \mathcal{N}(0,\,\sigma^{2})$ from the case where we have no measurement error. This approximation is valid if $\sigma << \sigma_{y_i}$ and $\sigma << \sigma_{x_{ij}}$ which is reasonable to assume. Our dataset now is still the same $D=\{x_{i1}, \ldots, x_{ip}, y_i\}_{i=1}^n$ but we have errors $D_\sigma = \{\sigma_{x_{i1}}, \ldots, \sigma_{x_{ip}}, \sigma_{y_i}\}_{i=1}^n$. Now the trick is to convert all the measurement errors into one $\sigma_i$, then it is just weighted regression. The conditional probability of one measurement is now a little bit more complex. Since now every $x_ij$ is sampled from prior probability distribution $\mathcal{N}(0,\,\sigma_{x_{ij}}^2)$ we have to account for that
$$P(\boldsymbol{\beta}| \mathbf{x}_i, y_i, \mathbf{\sigma_{x_i}}, \sigma_{y_i})
= P(\boldsymbol{\beta}| \mathbf{x}^\prime_i, y_i, \sigma_{y_i}) P(x^\prime_{i1} | x_{i1}, \sigma_{x_{ip}}) \ldots P(x^\prime_{ip} | x_{ip}, \sigma_{x_{ip}}).$$
That is in fact just the convolution of normal distributions
$$P(\boldsymbol{\beta}| \mathbf{x}_i, y_i, \mathbf{\sigma_{x_i}}, \sigma_{y_i})
= \int_{-\infty}^{+\infty} \cdots \int_{-\infty}^{+\infty} \frac{1}{\sqrt{2\pi\sigma_{y_i}^2}}\exp{\left(-\frac{(y_i - f(\mathbf{x}^\prime_i, \boldsymbol{\beta}))^2}{2\sigma_{y_i}^2}\right)} \frac{1}{\sqrt{2\pi\sigma_{x_{i1}}^2}}\exp{\left(-\frac{(x_{i1} - x^\prime_{i1})^2}{2\sigma_{x_{i1}}^2}\right)} \ldots $$
$$ \ldots \frac{1}{\sqrt{2\pi\sigma_{x_{ip}}^2}}\exp{\left(-\frac{(x_{ip} - x^\prime_{ip})^2}{2\sigma_{x_{ip}}^2}\right)} \,dx^\prime_{i1}\ldots\,dx^\prime_{ip}.$$
Which leads to a single normal distribution.
$$P(\boldsymbol{\beta}| \mathbf{x}_i, y_i, \sigma_i) = \frac{1}{\sqrt{2\pi\sigma_i^2}}\exp{\left(-\frac{(y_i - f(\mathbf{x}_i, \boldsymbol{\beta}))^2}{2\sigma_i^2}\right)},$$
where the new variance $\sigma_i^2$ is defined as
$$\sigma_i^2 = \sigma_{y_i}^2 + \beta_1^2 \sigma_{x_{i1}}^2 \ldots \beta_p^2 \sigma_{x_{ip}}^2$$
The rest is just weighted regression as we mentioned above
$$W_{ii} = \frac{1}{\sigma_i^2} = \frac{1}{\sigma_{y_i}^2 + \beta_1^2 \sigma_{x_{i1}}^2 \ldots \beta_p^2 \sigma_{x_{ip}}^2}$$
Maximizing loss function we will find optimal parameters $\boldsymbol{\beta}$, this leads to weighted least squares
$$\boldsymbol{\hat{\beta}} = \underset{\boldsymbol{\beta}}{\operatorname{argmin}} \sum_{i=1}^{N} \frac{1}{\sigma_i^2} (y_i - f(\mathbf{x}_i, \boldsymbol{\beta}))^2 $$
and the optimal solution is as before
$$\boldsymbol{\hat{\beta}} = \left(X^\textsf{T}WX\right)^{-1}X^\textsf{T}WY.$$

We can use this solution which is implemented in for example `np.polyfit` or we can implement the optimization ourselves with a model, objective function and jacobian function for `'Newton-CG` method and use the optimization function from `scipy.optimize.minimize`.

Sources:
* https://math.stackexchange.com/questions/1745174/convolution-of-two-gaussian-functions
* https://en.wikipedia.org/wiki/Convolution_of_probability_distributions
* https://math.stackexchange.com/questions/3839245/using-the-fourier-transform-to-prove-the-convolution-of-two-gaussians-is-gaussia

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

def model(x, a, b):
    return a * x + b

def objective_function(params, x, y, sigma_x, sigma_y):
    a, b = params
    weights = 1 / (sigma_y**2 + (a * sigma_x)**2)
    residuals = (y - model(x, a, b)) * np.sqrt(weights)
    return np.sum(residuals**2)

def jacobian_weighted(params, x, y, sigma_x, sigma_y):
    a, b = params
    weights = 1 / (sigma_y**2 + (a * sigma_x)**2)
    residues = y - model(x, a, b)
    da = 2 * np.sum(weights**2 * residues * (a*(b-y)*sigma_x**2 - x*sigma_y**2) )
    db = -2 * np.sum(weights * residues)
    return np.array([da, db])

a_true = 2
b_true = 1
x_data = np.arange(1, 11)

sigma_x = 0.3 * x_data 
sigma_y = 0.4 * x_data 
np.random.seed(0)
err_x = np.random.normal(0, sigma_x, x_data.shape)
err_y = np.random.normal(0, sigma_y, x_data.shape)
# we will create observed/measured data
x_obs = x_data + err_x
y_obs = a_true * x_data + b_true + err_y 

initial_guess = [0.5, 0.5]
result = minimize(
    fun=lambda params: objective_function(params, x_obs, y_obs, sigma_x, sigma_y),
    x0=initial_guess,
    method='Newton-CG',
    jac=lambda params: jacobian_weighted(params, x_obs, y_obs, sigma_x, sigma_y),
    options={'disp': True, 'xtol': 1e-8}
)
a_fit, b_fit = result.x
print(f"Parameter a: {a_true:.4f}, estimated: {a_fit:.4f}")
print(f"Parameter b: {b_true:.4f}, estimated: {b_fit:.4f}")

x_fit = np.linspace(min(x_data), max(x_data), 100)
y_fit = model(x_fit, a_fit, b_fit)
y_true = model(x_fit, a_true, b_true)

plt.errorbar(x_obs, y_obs, xerr=sigma_x, yerr=sigma_y, fmt='o', ecolor='gray', capsize=2, label='Noisy data')
plt.plot(x_fit, y_true, 'g--', label='True function')
plt.plot(x_fit, y_fit, 'r-', label='Fitted line')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

How do we estimate the error of fitted parameters when we have measurement error in $y$? We will again use the variance matrix of $\boldsymbol{\beta}$ leading to slightly different results. We define matrix $M$ as
$$M_{ii} = \sigma_{y_i}^2 + \beta_1^2 \sigma_{x_{i1}}^2 \ldots \beta_p^2 \sigma_{x_{ip}}^2.$$
Remember the definition of matrix $W$
$$W_{ii} = \frac{1}{\sigma_{y_i}^2 + \beta_1^2 \sigma_{x_{i1}}^2 \ldots \beta_p^2 \sigma_{x_{ip}}^2}.$$
The variance matrix for those optimal parameters is
$$\textrm{Var}(\boldsymbol{\hat{\beta}})
= (X^\textsf{T} W X)^{-1} X^\textsf{T} W M W X (X^\textsf{T} W X)^{-1}
= (X^\textsf{T} W X)^{-1} (X^\textsf{T} W X)  (X^\textsf{T} W X)^{-1}
= (X^\textsf{T} W X)^{-1}.$$

In [ ]:
from scipy.stats import t

residuals = y_obs - model(x_obs, a_fit, b_fit)
weights = 1 / (sigma_y**2 + (a_fit * sigma_x)**2)
W = np.diag(weights)
X_design = np.vstack((np.ones(len(x_obs)), x_obs)).T
cov_matrix = np.linalg.inv(X_design.T @ W @ X_design)
std_errors = np.sqrt(np.diag(cov_matrix))
df = len(x_obs) - len(initial_guess)

# t-value for 95% CI
t_value = t.ppf(1 - 0.025, df)
# 95% confidence intervals
ci_95 = t_value * std_errors
print(f"Standard error of a: ({a_fit - std_errors[1]:.4f}, {a_fit + std_errors[1]:.4f})")
print(f"95% CI for a: ({a_fit - ci_95[1]:.4f}, {a_fit + ci_95[1]:.4f})")

print(f"Standard error of b: ({b_fit - std_errors[0]:.4f}, {b_fit + std_errors[0]:.4f})")
print(f"95% CI for b: ({b_fit - ci_95[0]:.4f}, {b_fit + ci_95[0]:.4f})")

You may argue that adding weights to the loss function will not change anything. This section proposes that this is an important step and should not be ignored especially when the variance of measurement errors is great. Here we will optimize the parameters without weights and we can observe that the parameters are estimated worse than with weights.

In [ ]:
def objective_function(params, x, y):
    a, b = params
    residuals = y - model(x, a, b)
    return np.sum(residuals ** 2)

def jac_function(params, x, y):
    a, b = params
    residuals = y - model(x, a, b)
    da = -2 * np.sum(x * residuals)
    db = -2 * np.sum(residuals)
    return np.array([da, db])

initial_guess = [0.5, 0.5]
result = minimize(
    fun=lambda params: objective_function(params, x_obs, y_obs),
    x0=initial_guess,
    method='Newton-CG',
    jac=lambda params: jac_function(params, x_obs, y_obs),
    options={'disp': True, 'xtol': 1e-8}
)
a_fit, b_fit = result.x
print(f"Parameter a: {a_true:.4f}, estimated: {a_fit:.4f}")
print(f"Parameter b: {b_true:.4f}, estimated: {b_fit:.4f}")

x_fit = np.linspace(min(x_data), max(x_data), 100)
y_fit = model(x_fit, a_fit, b_fit)
y_true = model(x_fit, a_true, b_true)

plt.errorbar(x_obs, y_obs, xerr=sigma_x, yerr=sigma_y, fmt='o', ecolor='gray', capsize=2, label='Noisy data')
plt.plot(x_fit, y_true, 'g--', label='True function')
plt.plot(x_fit, y_fit, 'r-', label='Fitted line')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

<a id='deming_y_x'></a>
## Deming regression with measurement errors in $y$ and $x_i$

We no longer assume error of the model $\varepsilon \sim \mathcal{N}(0,\,\sigma^{2})$ from the case where we have no measurement error. This approximation is valid if $\sigma << \sigma_{y_i}$ and $\sigma << \sigma_{x_{ij}}$ which is reasonable to assume. In the previous section, we used measurement errors for $x_i$ to correctly estimate parameters $\boldsymbol{\beta}$ and use those errors to propagate them into the errors of estimated parameters $\boldsymbol{\beta}$. However one might argue that since we have errors on independent variables $x_i$ we can not use the measured values of $x_i$ as it will lead to a worse estimate. The proper way would be to use this prior not only for error estimation but also for parameter estimation. How we can do that? Well, we can create dummy parameters $\eta_i$ that will substitute measured values $x_i$. Then we will optimize at the same time $\boldsymbol{\beta}$ and $\boldsymbol{\eta}$. I personally don't recommend this approach as the first one, it can help when your measurement errors are really huge and you don't have many measured values. As the number of observed values grows I think the advantage of this approach diminishes. 

You can read about derivation [here](https://docplayer.net/236046756-Deming-regression-methcomp-package-may.html). 

Source:
* https://en.wikipedia.org/wiki/Deming_regression
* https://en.wikipedia.org/wiki/Total_least_squares
* https://docplayer.net/236046756-Deming-regression-methcomp-package-may.html

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

def model(x, a, b):
    return a * x + b

def objective_function(params, x, y, sigma_x, sigma_y):
    a, b = params[0:2]
    eta = params[2:]
    loss = np.sum((x - eta)**2 / sigma_x**2) 
    loss += np.sum((y - model(eta, a, b))**2 / sigma_y**2)
    return loss

def jacobian_weighted(params, x, y, sigma_x, sigma_y):
    a, b = params[0:2]
    eta = params[2:]
    weights = 1 / (sigma_y**2 + (a * sigma_x)**2)
    residues = y - model(eta, a, b)
    da = -2 * np.sum(residues / sigma_y**2 * eta)
    db = -2 * np.sum(residues / sigma_y**2)
    deta = - 2 * (x - eta) / sigma_x**2 - 2 * a*residues / sigma_y**2
    return np.array([da, db]+list(deta))

a_true = 2
b_true = 1
x_data = np.arange(1, 11)

sigma_x = 0.3 * x_data 
sigma_y = 0.4 * x_data 
np.random.seed(0)
err_x = np.random.normal(0, sigma_x, x_data.shape)
err_y = np.random.normal(0, sigma_y, x_data.shape)
# we will create observed/measured data
x_obs = x_data + err_x
y_obs = a_true * x_data + b_true + err_y 

initial_guess = [0.5, 0.5] + list(x_obs)
result = minimize(
    fun=lambda params: objective_function(params, x_obs, y_obs, sigma_x, sigma_y),
    x0=initial_guess,
    method='Newton-CG',
    jac=lambda params: jacobian_weighted(params, x_obs, y_obs, sigma_x, sigma_y),
    options={'disp': True, 'xtol': 1e-8}
)
a_fit, b_fit = result.x[0:2]
eta_fit = result.x[2:]
print(f"Parameter a: {a_true:.4f}, estimated: {a_fit:.4f}")
print(f"Parameter b: {b_true:.4f}, estimated: {b_fit:.4f}")
for i in range(len(x_data)):
    print(f"Parameter x_{i+1}: {x_data[i]:.4f}, estimated: {eta_fit[i]:.4f}")

x_fit = np.linspace(min(x_data), max(x_data), 100)
y_fit = model(x_fit, a_fit, b_fit)
y_true = model(x_fit, a_true, b_true)

plt.errorbar(x_obs, y_obs, xerr=sigma_x, yerr=sigma_y, fmt='o', ecolor='gray', capsize=2, label='Noisy data')
plt.plot(x_fit, y_true, 'g--', label='True function')
plt.plot(x_fit, y_fit, 'r-', label='Fitted line')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

As before we will use the weight matrix $W$
$$W_{ii} = \frac{1}{\sigma_{y_i}^2 + \beta_1^2 \sigma_{x_{i1}}^2 \ldots \beta_p^2 \sigma_{x_{ip}}^2}.$$
However in this case we no longer have linear regression. We can deal with this by using estimates for parameter errors using Jacobian. This approach is only an approximation
$$\textrm{Var}(\boldsymbol{\hat{\beta}})
\approx (J^\textsf{T} W J)^{-1}.$$
In this specific case, we will use only part of Jacobian as this error estimation is not numerically stable.

Source:
* https://en.wikipedia.org/wiki/Nonlinear_regression
* Jennrich, R. I. (1969). Asymptotic Properties of Non-Linear Least Squares Estimators. The Annals of Mathematical Statistics, 40(2), 633–643. http://www.jstor.org/stable/2239482

In [ ]:
from scipy.stats import t

residuals = y_obs - model(x_obs, a_fit, b_fit)
weights = 1 / (sigma_y**2 + (a_fit * sigma_x)**2)
W = np.diag(weights)
residues = y_obs - model(eta_fit, a_fit, b_fit)
J = np.vstack((np.ones(len(x_obs)), eta_fit)).T
cov_matrix = np.linalg.inv(J.T @ W @ J)
std_errors = np.sqrt(np.diag(cov_matrix))
df = len(x_obs) - 2

# t-value for 95% CI
t_value = t.ppf(1 - 0.025, df)
# 95% confidence intervals
ci_95 = t_value * std_errors
print(f"Standard error of a: ({a_fit - std_errors[1]:.4f}, {a_fit + std_errors[1]:.4f})")
print(f"95% CI for a: ({a_fit - ci_95[1]:.4f}, {a_fit + ci_95[1]:.4f})")

print(f"Standard error of b: ({b_fit - std_errors[0]:.4f}, {b_fit + std_errors[0]:.4f})")
print(f"95% CI for b: ({b_fit - ci_95[0]:.4f}, {b_fit + ci_95[0]:.4f})")

<a id='nonlin_y'></a>
## Nonlinear regression with measurement errors in $y$ 

The approach in this section is the same as in case of linear regression with measurement errors in $y$. The problem can be easily converted to weighted regression. We no longer assume error of the model $\varepsilon \sim \mathcal{N}(0,\,\sigma^{2})$ from the case where we have no measurement error. This approximation is valid if $\sigma << \sigma_{y_i}$ which is reasonable to assume. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

def model(x, A, omega, phi):
    return A*np.cos(2*np.pi*omega*(x+phi))

def objective_function(params, x, y, sigma_y):
    A, omega, phi = params
    loss = np.sum((y - model(x, A, omega, phi))**2 / sigma_y**2)
    return loss

def jacobian_weighted(params, x, y, sigma_y):
    A, omega, phi = params
    residues = y - model(x, A, omega, phi)
    dA = -2 * np.sum(residues / sigma_y**2 * np.cos(2*np.pi*omega*(x+phi)))
    domega = -2 * np.sum(residues / sigma_y**2 * (-A*np.sin(2*np.pi*omega*(x+phi))) * 2*np.pi*(x+phi))
    dphi = -2 * np.sum(residues / sigma_y**2 * (-A*np.sin(2*np.pi*omega*(x+phi))) * 2*np.pi*omega)
    return np.array([dA, domega, dphi])

A_true = 10
omega_true = 1/30.
phi_true = 3
x_data = np.arange(1, 31)

sigma_y = 0.007 * (x_data - 1/omega_true/2)**2 + 1
np.random.seed(0)
err_y = np.random.normal(0, sigma_y, x_data.shape)
# we will create observed/measured data
x_obs = x_data 
y_obs = model(x_obs, A_true, omega_true, phi_true) + err_y 

initial_guess = [20, 1/40., 3]
result = minimize(
    fun=lambda params: objective_function(params, x_obs, y_obs, sigma_y),
    x0=initial_guess,
    method='Newton-CG',
    jac=lambda params: jacobian_weighted(params, x_obs, y_obs, sigma_y),
    options={'disp': True, 'xtol': 1e-8}
)
A_fit, omega_fit, phi_fit = result.x
print(f"Parameter A: {A_true:.4f}, estimated: {A_fit:.4f}")
print(f"Parameter omega: {omega_true:.4f}, estimated: {omega_fit:.4f}")
print(f"Parameter phi: {phi_true:.4f}, estimated: {phi_fit:.4f}")

x_fit = np.linspace(min(x_data), max(x_data), 100)
y_fit = model(x_fit, A_fit, omega_fit, phi_fit)
y_true = model(x_fit, A_true, omega_true, phi_true)

plt.errorbar(x_obs, y_obs, yerr=sigma_y, fmt='o', ecolor='gray', capsize=2, label='Noisy data')
plt.plot(x_fit, y_true, 'g--', label='True function')
plt.plot(x_fit, y_fit, 'r-', label='Fitted line')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

As before we will use the weight matrix $W$
$$W_{ii} = 1/\sigma_{y_i}^2.$$
Instead of $X$ we will use in the nonlinear case Jacobian $J$
$$J_{ij} = \frac{\partial f(\boldsymbol{x_i}, \boldsymbol{\beta})}{\partial \beta_j}.$$
In our scenario
$$f(x_i, a, b) = A\cos (2 \pi \omega (x_i + \varphi))$$
$$J_{i1} = \cos (2 \pi \omega (x_i + \varphi))$$
$$J_{i2} = -2 \pi A\sin (2 \pi \omega (x_i + \varphi)) (x_i + \varphi)$$
$$J_{i3} = -2 \pi A \cos (2 \pi \omega (x_i + \varphi)) \omega $$
and remember to plug in the estimated parameters.
Variance matrix of parameters $\boldsymbol{\hat{\beta}}$ is estimated as
$$\textrm{Var}(\boldsymbol{\hat{\beta}})
\approx (J^\textsf{T} W J)^{-1}.$$

Source:
* https://en.wikipedia.org/wiki/Nonlinear_regression
* Jennrich, R. I. (1969). Asymptotic Properties of Non-Linear Least Squares Estimators. The Annals of Mathematical Statistics, 40(2), 633–643. http://www.jstor.org/stable/2239482

In [ ]:
from scipy.stats import t

residuals = y_obs - model(x_obs, A_fit, omega_fit, phi_fit)
weights = 1 / sigma_y**2 
W = np.diag(weights)
J = np.zeros((len(x_data), 3))
J[:, 0] = np.cos(2*np.pi*omega_fit*(x_obs+phi_fit))
J[:, 1] = (-A_fit*np.sin(2*np.pi*omega_fit*(x_obs+phi_fit))) * 2*np.pi*(x_obs+phi_fit)
J[:, 2] = (-A_fit*np.sin(2*np.pi*omega_fit*(x_obs+phi_fit))) * 2*np.pi*omega_fit
cov_matrix = np.linalg.inv(J.T @ W @ J)
std_errors = np.sqrt(np.diag(cov_matrix))
df = len(x_obs) - 2

# t-value for 95% CI
t_value = t.ppf(1 - 0.025, df)
# 95% confidence intervals
ci_95 = t_value * std_errors
print(f"Standard error of A: ({A_fit - std_errors[0]:.4f}, {A_fit + std_errors[0]:.4f})")
print(f"95% CI for A: ({A_fit - ci_95[0]:.4f}, {A_fit + ci_95[0]:.4f})")

print(f"Standard error of omega: ({omega_fit - std_errors[1]:.4f}, {omega_fit + std_errors[1]:.4f})")
print(f"95% CI for omega: ({omega_fit - ci_95[1]:.4f}, {omega_fit + ci_95[1]:.4f})")

print(f"Standard error of phi: ({phi_fit - std_errors[2]:.4f}, {phi_fit + std_errors[2]:.4f})")
print(f"95% CI for phi: ({phi_fit - ci_95[2]:.4f}, {phi_fit + ci_95[2]:.4f})")

<a id='nonlin_y_x'></a>
## Nonlinear regression with measurement errors in $y$ and $x_i$

This part is only an approximation that I came up with. We no longer assume error of the model $\varepsilon \sim \mathcal{N}(0,\,\sigma^{2})$ from the case where we have no measurement error. This approximation is valid if $\sigma << \sigma_{y_i}$ and $\sigma << \sigma_{x_{ij}}$ which is reasonable to assume.  We still would like to estimate the weights $1/\sigma_i^2$, but due to the nonlinear nature of our objective function, we can't perform simple convolution of normal distributions as the error term of $y_i$ is no longer sampled from a normal distribution. However, if we approximate this nonlinear transformation as linear, we can use the same approximation as in weighted regression. We can define this approximation as
$$\sigma_i^2 \approx \sigma_{y_i}^2 + \sigma_{x_{i}}^{\prime2}$$
where $\sigma_{x_{ij}}^\prime$ is defined as
$$x_i^+ = x_i + \sigma_x, y_i^+ = f(x_i^+, \boldsymbol{\beta})$$
$$x_i^- = x_i - \sigma_x, y_i^- = f(x_i^-, \boldsymbol{\beta})$$
$$\sigma_{x_{i}}^\prime = \frac{1}{2}|y_i^+ - y_i^-|$$

After this approximation, the rest is the same as in the section with linear regression with measurement errors in $y$ and $x_i$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

def model(x, A, omega, phi):
    return A*np.cos(2*np.pi*omega*(x+phi))

def objective_function(params, x, y, sigma_y):
    A, omega, phi = params
    loss = np.sum((y - model(x, A, omega, phi))**2 / sigma_y**2)
    return loss

def jacobian_weighted(params, x, y, sigma_y):
    A, omega, phi = params
    residues = y - model(x, A, omega, phi)
    dA = -2 * np.sum(residues / sigma_y**2 * np.cos(2*np.pi*omega*(x+phi)))
    domega = -2 * np.sum(residues / sigma_y**2 * (-A*np.sin(2*np.pi*omega*(x+phi))) * 2*np.pi*(x+phi))
    dphi = -2 * np.sum(residues / sigma_y**2 * (-A*np.sin(2*np.pi*omega*(x+phi))) * 2*np.pi*omega)
    return np.array([dA, domega, dphi])

def objective_function2(params, x, y, sigma_x, sigma_y):
    A, omega, phi = params
    x_plus, x_minus = x + sigma_x, x - sigma_x
    y_plus, y_minus = model(x_plus, A, omega, phi), model(x_minus, A, omega, phi)
    sigma_x_est = (y_plus - y_minus) / 2
    weight = 1 / (sigma_y**2 + sigma_x_est**2)
    loss = np.sum((y - model(x, A, omega, phi))**2 * weight)
    return loss

def jacobian_weighted2(params, x, y, sigma_x, sigma_y):
    A, omega, phi = params
    x_plus, x_minus = x + sigma_x, x - sigma_x
    y_plus, y_minus = model(x_plus, A, omega, phi), model(x_minus, A, omega, phi)
    sigma_x_est = (y_plus - y_minus) / 2
    weight = 1 / (sigma_y**2 + sigma_x_est**2)
    y_hat = model(x, A, omega, phi)
    residues = y - y_hat
    dA = -2 * np.sum(residues * weight * np.cos(2*np.pi*omega*(x+phi)))
    domega = -2 * np.sum(residues * weight * (-A*np.sin(2*np.pi*omega*(x+phi))) * 2*np.pi*(x+phi))
    dphi = -2 * np.sum(residues * weight * (-A*np.sin(2*np.pi*omega*(x+phi))) * 2*np.pi*omega)
    return np.array([dA, domega, dphi])

A_true = 10
omega_true = 1/30.
phi_true = 3
x_data = np.arange(1, 31)

sigma_y = 0.007 * (x_data - 1/omega_true/2)**2 + 1
sigma_x = 0.003 * (x_data - 1/omega_true/2)**2 + 0.5
np.random.seed(0)
err_y = np.random.normal(0, sigma_y, x_data.shape)
err_x = np.random.normal(0, sigma_x, x_data.shape)
# we will create observed/measured data
x_obs = x_data + err_x
y_obs = model(x_obs, A_true, omega_true, phi_true) + err_y 

# this parameter estimation will find wrong local minimum if we use weights with x_i error
# first we will assume weights with only y error
initial_guess = [15, 1/40., 0.5]
result = minimize(
    fun=lambda params: objective_function(params, x_obs, y_obs, sigma_y),
    x0=initial_guess,
    method='Newton-CG',
    jac=lambda params: jacobian_weighted(params, x_obs, y_obs, sigma_y),
    options={'disp': True, 'xtol': 1e-8}
)
A_fit, omega_fit, phi_fit = result.x

# with estimate that is probably close to optimal, we will use both y and x_i errors
initial_guess = [A_fit, omega_fit, phi_fit]
result = minimize(
    fun=lambda params: objective_function2(params, x_obs, y_obs, sigma_x, sigma_y),
    x0=initial_guess,
    method='Newton-CG',
    jac=lambda params: jacobian_weighted2(params, x_obs, y_obs, sigma_x, sigma_y),
    options={'disp': True, 'xtol': 1e-8}
)
A_fit, omega_fit, phi_fit = result.x
print(f"Parameter A: {A_true:.4f}, estimated: {A_fit:.4f}")
print(f"Parameter omega: {omega_true:.4f}, estimated: {omega_fit:.4f}")
print(f"Parameter phi: {phi_true:.4f}, estimated: {phi_fit:.4f}")

x_fit = np.linspace(min(x_data), max(x_data), 100)
y_fit = model(x_fit, A_fit, omega_fit, phi_fit)
y_true = model(x_fit, A_true, omega_true, phi_true)

plt.errorbar(x_obs, y_obs, xerr=sigma_x, yerr=sigma_y, fmt='o', ecolor='gray', capsize=2, label='Noisy data')
plt.plot(x_fit, y_true, 'g--', label='True function')
plt.plot(x_fit, y_fit, 'r-', label='Fitted line')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

As before we will use the weight matrix $W$
$$W_{ii} = \frac{1}{\sigma_{y_i}^2 + \sigma_{x_i}^{\prime2}},$$
where $\sigma_{x_i}^{\prime}$ is defined above.
Instead of $X$ we will use in the nonlinear case Jacobian $J$
$$J_{ij} = \frac{\partial f(\boldsymbol{x_i}, \boldsymbol{\beta})}{\partial \beta_j}.$$
In our scenario
$$f(x_i, a, b) = A\cos (2 \pi \omega (x_i + \varphi))$$
$$J_{i1} = \cos (2 \pi \omega (x + \varphi))$$
$$J_{i2} = -2 \pi A\sin (2 \pi \omega (x_i + \varphi)) (x_i + \varphi)$$
$$J_{i3} = -2 \pi A \cos (2 \pi \omega (x_i + \varphi)) \omega $$
and remember to plug in the estimated parameters.
Variance matrix of parameters $\boldsymbol{\hat{\beta}}$ is estimated as
$$\textrm{Var}(\boldsymbol{\hat{\beta}})
\approx (J^\textsf{T} W J)^{-1}.$$

Source:
* https://en.wikipedia.org/wiki/Nonlinear_regression
* Jennrich, R. I. (1969). Asymptotic Properties of Non-Linear Least Squares Estimators. The Annals of Mathematical Statistics, 40(2), 633–643. http://www.jstor.org/stable/2239482

In [ ]:
from scipy.stats import t

residuals = y_obs - model(x_obs, A_fit, omega_fit, phi_fit)
x_plus, x_minus = x_obs + sigma_x, x_obs - sigma_x
y_plus, y_minus = model(x_plus, A_fit, omega_fit, phi_fit), model(x_minus, A_fit, omega_fit, phi_fit)
sigma_x_est = (y_plus - y_minus) / 2
weights = 1 / (sigma_y**2 + sigma_x_est**2)
W = np.diag(weights)
J = np.zeros((len(x_data), 3))
J[:, 0] = np.cos(2*np.pi*omega_fit*(x_obs+phi_fit))
J[:, 1] = (-A_fit*np.sin(2*np.pi*omega_fit*(x_obs+phi_fit))) * 2*np.pi*(x_obs+phi_fit)
J[:, 2] = (-A_fit*np.sin(2*np.pi*omega_fit*(x_obs+phi_fit))) * 2*np.pi*omega_fit
cov_matrix = np.linalg.inv(J.T @ W @ J)
std_errors = np.sqrt(np.diag(cov_matrix))
df = len(x_obs) - 3

# t-value for 95% CI
t_value = t.ppf(1 - 0.025, df)
# 95% confidence intervals
ci_95 = t_value * std_errors
print(f"Standard error of A: ({A_fit - std_errors[0]:.4f}, {A_fit + std_errors[0]:.4f})")
print(f"95% CI for A: ({A_fit - ci_95[0]:.4f}, {A_fit + ci_95[0]:.4f})")

print(f"Standard error of omega: ({omega_fit - std_errors[1]:.4f}, {omega_fit + std_errors[1]:.4f})")
print(f"95% CI for omega: ({omega_fit - ci_95[1]:.4f}, {omega_fit + ci_95[1]:.4f})")

print(f"Standard error of phi: ({phi_fit - std_errors[2]:.4f}, {phi_fit + std_errors[2]:.4f})")
print(f"95% CI for phi: ({phi_fit - ci_95[2]:.4f}, {phi_fit + ci_95[2]:.4f})")

<a id='mc_y_x'></a>
## Monte Carlo estimation of parameters errors with measurement errors for $y$ and $x_i$

We no longer assume error of the model $\varepsilon \sim \mathcal{N}(0,\,\sigma^{2})$ from the case where we have no measurement error. This approximation is valid if $\sigma << \sigma_{y_i}$ and $\sigma << \sigma_{x_{ij}}$ which is reasonable to assume. There is one approach which is slow, but more robust. When defining the probability of parameters given the dataset with errors
$$P(\boldsymbol{\beta}| \mathbf{x}_i, y_i, \mathbf{\sigma_{x_i}}, \sigma_{y_i})
= P(\boldsymbol{\beta}| \mathbf{x}^\prime_i, y_i, \sigma_{y_i}) P(x^\prime_{i1} | x_{i1}, \sigma_{x_{ip}}) \ldots P(x^\prime_{ip} | x_{ip}, \sigma_{x_{ip}}),$$
we decided to calculate this convolution integral directly. There is another approach, where we can instead of $P(\boldsymbol{\beta}| \mathbf{x}_i, y_i, \mathbf{\sigma_{x_i}}, \sigma_{y_i})$ use directly probability $P(\boldsymbol{\beta}| \mathbf{x}_i^\prime, y_i, \sigma_{y_i})$ estimate parameters and parameter erros as in linear regression in $y$ and $x_i$. Repeat this process while we will sample again and again from prior distributions for $x_{ij}^\prime \sim \mathcal{N}(0,\,\sigma_{x_{ij}}^2)$ and we will save all estimated parameters. We then will use the sample of our new estimated parameters to get $\sigma_{\beta_i}$ as
$$\sigma_{\beta_i}^2 = Var(\{\beta_{i1}, \ldots \beta_{iN}\}).$$
This approach has only one drawback and the error of such a method is proportional to $1/\sqrt{N}$, so we need typically pretty high $N$ to observe convergence. The estimated error that we could calculate in every step of Monte Carlo should not be used in any way. This error is already accounted for when estimating $\sigma_{\beta_i}^2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from tqdm import tqdm

def model(x, A, omega, phi):
    return A*np.cos(2*np.pi*omega*(x+phi))

def objective_function(params, x, y, sigma_y):
    A, omega, phi = params
    loss = np.sum((y - model(x, A, omega, phi))**2 / sigma_y**2)
    return loss

def jacobian_weighted(params, x, y, sigma_y):
    A, omega, phi = params
    residues = y - model(x, A, omega, phi)
    dA = -2 * np.sum(residues / sigma_y**2 * np.cos(2*np.pi*omega*(x+phi)))
    domega = -2 * np.sum(residues / sigma_y**2 * (-A*np.sin(2*np.pi*omega*(x+phi))) * 2*np.pi*(x+phi))
    dphi = -2 * np.sum(residues / sigma_y**2 * (-A*np.sin(2*np.pi*omega*(x+phi))) * 2*np.pi*omega)
    return np.array([dA, domega, dphi])

def objective_function2(params, x, y, sigma_x, sigma_y):
    A, omega, phi = params
    x_plus, x_minus = x + sigma_x, x - sigma_x
    y_plus, y_minus = model(x_plus, A, omega, phi), model(x_minus, A, omega, phi)
    sigma_x_est = (y_plus - y_minus) / 2
    weight = 1 / (sigma_y**2 + sigma_x_est**2)
    loss = np.sum((y - model(x, A, omega, phi))**2 * weight)
    return loss

def jacobian_weighted2(params, x, y, sigma_x, sigma_y):
    A, omega, phi = params
    x_plus, x_minus = x + sigma_x, x - sigma_x
    y_plus, y_minus = model(x_plus, A, omega, phi), model(x_minus, A, omega, phi)
    sigma_x_est = (y_plus - y_minus) / 2
    weight = 1 / (sigma_y**2 + sigma_x_est**2)
    y_hat = model(x, A, omega, phi)
    residues = y - y_hat
    dA = -2 * np.sum(residues * weight * np.cos(2*np.pi*omega*(x+phi)))
    domega = -2 * np.sum(residues * weight * (-A*np.sin(2*np.pi*omega*(x+phi))) * 2*np.pi*(x+phi))
    dphi = -2 * np.sum(residues * weight * (-A*np.sin(2*np.pi*omega*(x+phi))) * 2*np.pi*omega)
    return np.array([dA, domega, dphi])

A_true = 10
omega_true = 1/30.
phi_true = 3
x_data = np.arange(1, 31)

sigma_x = 0.003 * (x_data - 1/omega_true/2)**2 + 0.5
sigma_y = 0.007 * (x_data - 1/omega_true/2)**2 + 1
np.random.seed(0)
err_y = np.random.normal(0, sigma_y, x_data.shape)
err_x = np.random.normal(0, sigma_x, x_data.shape)
# we will create observed/measured data
x_obs = x_data + err_x
y_obs = model(x_obs, A_true, omega_true, phi_true) + err_y 

# this parameter estimation will find wrong local minimum if we use weights with x_i error
# first we will assume weights with only y error
initial_guess = [15, 1/40., 0.5]
result = minimize(
    fun=lambda params: objective_function(params, x_obs, y_obs, sigma_y),
    x0=initial_guess,
    method='Newton-CG',
    jac=lambda params: jacobian_weighted(params, x_obs, y_obs, sigma_y),
    options={'disp': True, 'xtol': 1e-8}
)
A_fit, omega_fit, phi_fit = result.x

initial_guess = [A_fit, omega_fit, phi_fit]
params_sampled = []
for i in tqdm(range(10**4)):
    # here we sample new data given the errors for y and x_i
    err_y_sampling = np.random.normal(0, sigma_y, x_data.shape)
    err_x_sampling = np.random.normal(0, sigma_x, x_data.shape)
    x_sampled = x_obs + err_x_sampling
    y_sampled = y_obs + err_y_sampling
    # with estimate that is probably close to optimal, we will use both y and x_i errors
    result = minimize(
        fun=lambda params: objective_function2(params, x_sampled, y_sampled, sigma_x, sigma_y),
        x0=initial_guess,
        method='Newton-CG',
        jac=lambda params: jacobian_weighted2(params, x_sampled, y_sampled, sigma_x, sigma_y),
        options={'xtol': 1e-8, 'maxiter': 10**4}
    )
    A_fit, omega_fit, phi_fit = result.x
    params_sampled.append([A_fit, omega_fit, phi_fit])
params_sampled = np.array(params_sampled)
A_fit, A_err = np.mean(params_sampled[:, 0]), np.std(params_sampled[:, 0])
omega_fit, omega_err = np.mean(params_sampled[:, 1]), np.std(params_sampled[:, 1])
phi_fit, phi_err = np.mean(params_sampled[:, 2]), np.std(params_sampled[:, 2])
print(f"Parameter A: {A_true:.4f}, estimated: {A_fit:.4f}")
print(f"Parameter omega: {omega_true:.4f}, estimated: {omega_fit:.4f}")
print(f"Parameter phi: {phi_true:.4f}, estimated: {phi_fit:.4f}")

plt.scatter(params_sampled[:, 0], params_sampled[:, 1], alpha=0.5, s=3)
plt.xlabel("A")
plt.ylabel("omega")
plt.show()

plt.scatter(params_sampled[:, 0], params_sampled[:, 2], alpha=0.5, s=3)
plt.xlabel("A")
plt.ylabel("phi")
plt.show()

plt.scatter(params_sampled[:, 1], params_sampled[:, 2], alpha=0.5, s=3)
plt.xlabel("omega")
plt.ylabel("phi")
plt.show()

x_fit = np.linspace(min(x_data), max(x_data), 100)
y_fit = model(x_fit, A_fit, omega_fit, phi_fit)
y_true = model(x_fit, A_true, omega_true, phi_true)

plt.errorbar(x_obs, y_obs, xerr=sigma_x, yerr=sigma_y, fmt='o', ecolor='gray', capsize=2, label='Noisy data')
plt.plot(x_fit, y_true, 'g--', label='True function')
plt.plot(x_fit, y_fit, 'r-', label='Fitted line')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

And finally the estimated erorrs.

In [ ]:
from scipy.stats import t

std_errors = np.array([A_err, omega_err, phi_err])
df = len(x_obs) - 3
# t-value for 95% CI
t_value = t.ppf(1 - 0.025, df)
# 95% confidence intervals
ci_95 = t_value * std_errors
print(f"Standard error of A: ({A_fit - std_errors[0]:.4f}, {A_fit + std_errors[0]:.4f})")
print(f"95% CI for A: ({A_fit - ci_95[0]:.4f}, {A_fit + ci_95[0]:.4f})")

print(f"Standard error of omega: ({omega_fit - std_errors[1]:.4f}, {omega_fit + std_errors[1]:.4f})")
print(f"95% CI for omega: ({omega_fit - ci_95[1]:.4f}, {omega_fit + ci_95[1]:.4f})")

print(f"Standard error of phi: ({phi_fit - std_errors[2]:.4f}, {phi_fit + std_errors[2]:.4f})")
print(f"95% CI for phi: ({phi_fit - ci_95[2]:.4f}, {phi_fit + ci_95[2]:.4f})")

<a id='cite'></a>
## How to cite

If you want to reference this work you can do it by
```text
@misc{daniel_herman_2024,
    title={Measurement error in regression},
    url={https://www.kaggle.com/code/jetakow/measurement-error-in-regression},
    publisher={Kaggle},
    author={Daniel Herman},
    year={2024}
}
```
DOI is only generated for kaggle datasets. Or inline
```text
Herman, D. (2024, May). Measurement error in regression}, Version 10. Retrieved [Date Retrieved] from https://www.kaggle.com/code/jetakow/measurement-error-in-regression.
```
make sure you change the version to the latest.

Source: https://www.kaggle.com/discussions/general/46091

Hope this article will help you to correctly propagate errors. If you have any questions leave a comment on you can reach me via email `daniel dot herman at protonmail dot com`.